# SIEWS+ Rust & Corrosion Detection Training

Notebook ini menjalankan fine-tuning YOLO untuk dataset `Rust Corrosion Detection.v16i.yolo26` dengan guardrail sebelum training:

1. Deteksi path lokal/Kaggle tanpa hard-code drive Windows.
2. Audit split, class id, missing/orphan label, dan bbox invalid.
3. Opsional remap class id jika audit visual membuktikan label tertukar.
4. Buat prepared dataset bersih jika ada row label invalid.
5. Tulis `rust_corrosion.clean.yaml`, train, evaluasi, preview inference, lalu export `.pt`.

Class map canonical:
- `0: corrosion`
- `1: moderate corrosion`
- `2: rust`
- `3: severe corrosion`

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import yaml

IS_KAGGLE = Path('/kaggle/input').exists()
print(f'Kaggle runtime: {IS_KAGGLE}')

try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'ultralytics', '-q'], check=True)
    import ultralytics

import torch
from ultralytics import YOLO

print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
SEED = 42
random.seed(SEED)

## 2. Konfigurasi Path dan Hyperparameter

In [ ]:
FOLDER_NAME = 'Rust Corrosion Detection.v16i.yolo26'
ZIP_NAME = f'{FOLDER_NAME}.zip'
RUN_NAME = 'rust_corrosion'
EXPORT_NAME = 'rust_corrosion.pt'

CLASS_NAMES = {
    0: 'corrosion',
    1: 'moderate corrosion',
    2: 'rust',
    3: 'severe corrosion',
}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Training defaults. Ubah di sini sebelum menjalankan cell training.
EPOCHS = 100
BATCH = 16
IMGSZ = 640
PATIENCE = 25
FREEZE = 0
CONF = 0.25
SAMPLE_SIZE = 12

# Isi hanya jika audit visual membuktikan class id tertukar, contoh: {0: 3, 3: 0}
REMAP = {}
CLEAN_INVALID_LABELS = True
SKIP_PREVIEW = False


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'training').exists() and (candidate / 'dataset').exists():
            return candidate
    return start.resolve()

ROOT_DIR = Path('/kaggle/working') if IS_KAGGLE else find_repo_root(Path.cwd())
RUN_ROOT = Path('/kaggle/working/runs_rust_corrosion') if IS_KAGGLE else ROOT_DIR / 'training' / 'runs_rust_corrosion'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUN_ROOT / 'runs' / RUN_NAME

LOCAL_DATASET_DIR = ROOT_DIR / 'dataset' / FOLDER_NAME
LOCAL_ZIP = ROOT_DIR / 'dataset' / ZIP_NAME

print(f'ROOT_DIR : {ROOT_DIR}')
print(f'RUN_ROOT : {RUN_ROOT}')
print(f'DEVICE   : {DEVICE}')

## 3. Dataset Discovery

In [ ]:
def looks_like_yolo_dataset(path: Path) -> bool:
    return (path / 'train' / 'images').exists() and (path / 'train' / 'labels').exists()


def find_kaggle_dataset() -> Path:
    input_dir = Path('/kaggle/input')
    candidates = []
    for path in input_dir.rglob('*'):
        if path.is_dir() and looks_like_yolo_dataset(path):
            candidates.append(path)
    if candidates:
        candidates = sorted(candidates, key=lambda p: (FOLDER_NAME.lower() not in str(p).lower(), len(str(p))))
        return candidates[0]

    zip_candidates = sorted(input_dir.rglob(ZIP_NAME))
    if not zip_candidates:
        zip_candidates = sorted(p for p in input_dir.rglob('*.zip') if 'rust' in p.name.lower() or 'corrosion' in p.name.lower())
    if zip_candidates:
        out_dir = RUN_ROOT / 'dataset_from_zip'
        if not looks_like_yolo_dataset(out_dir):
            out_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_candidates[0], 'r') as zf:
                zf.extractall(out_dir)
        nested = [p for p in out_dir.rglob('*') if p.is_dir() and looks_like_yolo_dataset(p)]
        return sorted(nested, key=lambda p: len(str(p)))[0] if nested else out_dir

    raise FileNotFoundError('Dataset YOLO rust/corrosion tidak ditemukan di /kaggle/input. Pastikan dataset sudah di-Add ke notebook Kaggle.')


def resolve_dataset_dir() -> Path:
    if IS_KAGGLE:
        return find_kaggle_dataset().resolve()
    if looks_like_yolo_dataset(LOCAL_DATASET_DIR):
        return LOCAL_DATASET_DIR.resolve()
    if LOCAL_ZIP.exists():
        out_dir = RUN_ROOT / 'dataset'
        if not looks_like_yolo_dataset(out_dir):
            out_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
                zf.extractall(out_dir)
        nested = [p for p in out_dir.rglob('*') if p.is_dir() and looks_like_yolo_dataset(p)]
        return sorted(nested, key=lambda p: len(str(p)))[0] if nested else out_dir.resolve()
    raise FileNotFoundError(f'Dataset tidak ditemukan: {LOCAL_DATASET_DIR} atau {LOCAL_ZIP}')

SOURCE_DATASET_DIR = resolve_dataset_dir()
print(f'Dataset source: {SOURCE_DATASET_DIR}')

for split in ['train', 'valid', 'val', 'test']:
    image_dir = SOURCE_DATASET_DIR / split / 'images'
    label_dir = SOURCE_DATASET_DIR / split / 'labels'
    if image_dir.exists():
        image_count = sum(1 for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
        label_count = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
        print(f'[{split:5s}] images={image_count:5d} labels={label_count:5d}')

## 4. Utility Audit Dataset

In [ ]:
def split_names(dataset_dir: Path) -> Tuple[str, str]:
    val_split = 'valid' if (dataset_dir / 'valid' / 'images').exists() else 'val'
    test_split = 'test' if (dataset_dir / 'test' / 'images').exists() else val_split
    return val_split, test_split


def iter_images(images_dir: Path) -> List[Path]:
    if not images_dir.exists():
        return []
    return sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def read_label_file(path: Path) -> List[Tuple[int, float, float, float, float]]:
    rows = []
    if not path.exists():
        return rows
    for line_no, line in enumerate(path.read_text().splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            raise ValueError(f'{path}:{line_no} has {len(parts)} columns, expected at least 5.')
        cls_id = int(float(parts[0]))
        coords = tuple(float(v) for v in parts[1:5])
        rows.append((cls_id, *coords))
    return rows


def is_valid_yolo_row(row: Tuple[int, float, float, float, float]) -> bool:
    cls_id, xc, yc, width, height = row
    return (
        cls_id in CLASS_NAMES
        and 0 <= xc <= 1
        and 0 <= yc <= 1
        and 0 < width <= 1
        and 0 < height <= 1
    )


def audit_dataset(dataset_dir: Path) -> Dict[str, Dict]:
    if not dataset_dir.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_dir}')

    val_split, test_split = split_names(dataset_dir)
    report = {}
    label_to_images = defaultdict(list)

    for split in ['train', val_split, test_split]:
        images_dir = dataset_dir / split / 'images'
        labels_dir = dataset_dir / split / 'labels'
        images = iter_images(images_dir)
        labels = sorted(labels_dir.glob('*.txt')) if labels_dir.exists() else []
        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        counts = Counter()
        empty = 0
        invalid_rows = []
        invalid_ids = Counter()

        for label_path in labels:
            try:
                rows = read_label_file(label_path)
            except Exception as exc:
                invalid_rows.append(str(exc))
                continue

            if not rows:
                empty += 1
            for cls_id, xc, yc, width, height in rows:
                counts[cls_id] += 1
                if cls_id not in CLASS_NAMES:
                    invalid_ids[cls_id] += 1
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < width <= 1 and 0 < height <= 1):
                    invalid_rows.append(f'{label_path}: invalid bbox cls={cls_id} xywh={xc:.4f},{yc:.4f},{width:.4f},{height:.4f}')
                if cls_id in CLASS_NAMES and len(label_to_images[cls_id]) < 12:
                    for ext in IMAGE_EXTENSIONS:
                        image_path = images_dir / f'{label_path.stem}{ext}'
                        if image_path.exists():
                            label_to_images[cls_id].append(image_path)
                            break

        report[split] = {
            'images': len(images),
            'labels': len(labels),
            'empty_labels': empty,
            'missing_labels': sorted(image_stems - label_stems)[:20],
            'orphan_labels': sorted(label_stems - image_stems)[:20],
            'counts': dict(sorted(counts.items())),
            'invalid_ids': dict(sorted(invalid_ids.items())),
            'invalid_rows': invalid_rows[:20],
        }

    report['_meta'] = {
        'dataset': str(dataset_dir),
        'val_split': val_split,
        'test_split': test_split,
        'class_names': CLASS_NAMES,
        'label_examples': {str(k): [str(p) for p in v] for k, v in label_to_images.items()},
    }
    return report


def report_has_invalid_labels(report: Dict[str, Dict]) -> bool:
    for split, info in report.items():
        if split.startswith('_'):
            continue
        if info['invalid_ids'] or info['invalid_rows']:
            return True
    return False


def print_audit(report: Dict[str, Dict]) -> None:
    print('\n=== Rust/Corrosion Dataset Audit ===')
    print(f"dataset: {report['_meta']['dataset']}")
    print('class map:')
    for cls_id, name in CLASS_NAMES.items():
        print(f'  {cls_id}: {name}')

    for split, info in report.items():
        if split.startswith('_'):
            continue
        total = sum(info['counts'].values())
        print(f"\n[{split}] images={info['images']} labels={info['labels']} empty={info['empty_labels']} instances={total}")
        for cls_id in sorted(CLASS_NAMES):
            count = info['counts'].get(cls_id, 0)
            pct = (count / total * 100) if total else 0.0
            marker = '  <-- no instances in this split' if count == 0 else ''
            print(f'  {cls_id} {CLASS_NAMES[cls_id]:20s}: {count:6d} ({pct:5.1f}%){marker}')
        if info['missing_labels']:
            print(f"  warning: sample images without labels: {info['missing_labels'][:3]}")
        if info['orphan_labels']:
            print(f"  warning: sample labels without images: {info['orphan_labels'][:3]}")
        if info['invalid_ids']:
            print(f"  error: invalid class ids: {info['invalid_ids']}")
        if info['invalid_rows']:
            print(f"  error: invalid rows: {info['invalid_rows'][:3]}")

## 5. Prepared Dataset dan YAML Clean

In [ ]:
def link_or_copy_images(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.symlink(src.resolve(), dst, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst)


def create_prepared_dataset(dataset_dir: Path, run_root: Path, remap: Dict[int, int], clean_invalid: bool) -> Path:
    if not remap and not clean_invalid:
        return dataset_dir

    parts = []
    if clean_invalid:
        parts.append('clean')
    if remap:
        parts.append('remap_' + '_'.join(f'{old}to{new}' for old, new in sorted(remap.items())))
    out_dir = run_root / ('dataset_' + '_'.join(parts))
    val_split, test_split = split_names(dataset_dir)

    print(f'\n[INFO] Creating prepared dataset: {out_dir}')
    if remap:
        print(f'       remap: {remap}')
    if clean_invalid:
        print('       cleaning: invalid class ids and zero-area/out-of-range boxes are removed')

    removed_rows = Counter()
    for split in ['train', val_split, test_split]:
        src_images = dataset_dir / split / 'images'
        src_labels = dataset_dir / split / 'labels'
        dst_split = out_dir / split
        dst_labels = dst_split / 'labels'
        dst_split.mkdir(parents=True, exist_ok=True)
        dst_labels.mkdir(parents=True, exist_ok=True)
        link_or_copy_images(src_images, dst_split / 'images')

        for src_label in src_labels.glob('*.txt'):
            new_lines = []
            for line_no, line in enumerate(src_label.read_text().splitlines(), start=1):
                if not line.strip():
                    continue
                parts = line.split()
                try:
                    if len(parts) < 5:
                        raise ValueError(f'expected at least 5 columns, got {len(parts)}')
                    old_id = int(float(parts[0]))
                    row = (remap.get(old_id, old_id), *(float(v) for v in parts[1:5]))
                except Exception:
                    if clean_invalid:
                        removed_rows[split] += 1
                        continue
                    raise ValueError(f'{src_label}:{line_no} cannot be parsed: {line}')

                parts[0] = str(row[0])
                if clean_invalid and not is_valid_yolo_row(row):
                    removed_rows[split] += 1
                    continue
                new_lines.append(' '.join(parts))
            (dst_labels / src_label.name).write_text(('\n'.join(new_lines) + '\n') if new_lines else '')

    if removed_rows:
        print(f'       removed rows: {dict(sorted(removed_rows.items()))}')
    return out_dir


def write_data_yaml(dataset_dir: Path, run_root: Path) -> Path:
    val_split, test_split = split_names(dataset_dir)
    yaml_path = run_root / 'rust_corrosion.clean.yaml'
    payload = {
        'path': str(dataset_dir.resolve()),
        'train': 'train/images',
        'val': f'{val_split}/images',
        'test': f'{test_split}/images',
        'nc': len(CLASS_NAMES),
        'names': [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)],
    }
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False))
    print(f'\n[OK] Clean YOLO yaml written: {yaml_path}')
    print(yaml_path.read_text())
    return yaml_path


def save_audit_report(report: Dict[str, Dict], run_root: Path, name: str = 'dataset_audit.json') -> Path:
    run_root.mkdir(parents=True, exist_ok=True)
    report_path = run_root / name
    report_path.write_text(json.dumps(report, indent=2))
    print(f'[OK] Audit report written: {report_path}')
    return report_path

## 6. Preview Helper

In [ ]:
def draw_yolo_boxes(image, rows: Sequence[Tuple[int, float, float, float, float]], class_names: Dict[int, str], title: str):
    import cv2

    canvas = image.copy()
    height, width = canvas.shape[:2]
    colors = {
        0: (255, 120, 20),
        1: (0, 210, 255),
        2: (255, 60, 60),
        3: (255, 230, 0),
    }
    for cls_id, xc, yc, bw, bh in rows:
        x1 = int((xc - bw / 2) * width)
        y1 = int((yc - bh / 2) * height)
        x2 = int((xc + bw / 2) * width)
        y2 = int((yc + bh / 2) * height)
        color = colors.get(cls_id, (255, 255, 255))
        label = class_names.get(cls_id, f'cls_{cls_id}')
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        cv2.putText(canvas, label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    cv2.putText(canvas, title, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return canvas


def save_label_preview(dataset_dir: Path, run_root: Path, sample_size: int = SAMPLE_SIZE, seed: int = SEED) -> Optional[Path]:
    try:
        import cv2
        import matplotlib.pyplot as plt
    except ImportError as exc:
        print(f'[WARN] Skipping label preview because optional package is missing: {exc}')
        return None

    image_dir = dataset_dir / 'train' / 'images'
    label_dir = dataset_dir / 'train' / 'labels'
    rng = random.Random(seed)
    examples = defaultdict(list)

    for label_path in sorted(label_dir.glob('*.txt')):
        rows = read_label_file(label_path)
        ids = {row[0] for row in rows if is_valid_yolo_row(row)}
        if not ids:
            continue
        image_path = None
        for ext in IMAGE_EXTENSIONS:
            candidate = image_dir / f'{label_path.stem}{ext}'
            if candidate.exists():
                image_path = candidate
                break
        if image_path is None:
            continue
        for cls_id in ids:
            if len(examples[cls_id]) < sample_size:
                examples[cls_id].append(image_path)

    selected = []
    per_class = max(1, min(3, sample_size // max(1, len(CLASS_NAMES))))
    for cls_id in sorted(CLASS_NAMES):
        candidates = examples.get(cls_id, [])
        if candidates:
            selected.extend((cls_id, p) for p in rng.sample(candidates, min(per_class, len(candidates))))

    if not selected:
        print('[WARN] No labels available for label preview.')
        return None

    cols = min(4, len(selected))
    rows_count = (len(selected) + cols - 1) // cols
    fig, axes = plt.subplots(rows_count, cols, figsize=(cols * 4.2, rows_count * 4.0))
    axes_flat = [axes] if rows_count == 1 and cols == 1 else list(getattr(axes, 'flat', axes))

    for ax, (target_cls_id, image_path) in zip(axes_flat, selected):
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            ax.axis('off')
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        rows = read_label_file(label_dir / f'{image_path.stem}.txt')
        canvas = draw_yolo_boxes(image_rgb, rows, CLASS_NAMES, f'ID {target_cls_id}: {CLASS_NAMES[target_cls_id]}')
        ax.imshow(canvas)
        ax.axis('off')
        ax.set_title(image_path.name[:50], fontsize=8)

    for ax in axes_flat[len(selected):]:
        ax.axis('off')

    out_png = run_root / 'label_audit_by_class.png'
    plt.tight_layout()
    fig.savefig(out_png, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'[OK] Label audit preview written: {out_png}')
    return out_png


def predictions_as_yolo_rows(result) -> List[Tuple[int, float, float, float, float]]:
    rows = []
    if result.boxes is None:
        return rows
    height, width = result.orig_shape[:2]
    for box in result.boxes:
        cls_id = int(box.cls[0])
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
        xc = ((x1 + x2) / 2) / width
        yc = ((y1 + y2) / 2) / height
        bw = (x2 - x1) / width
        bh = (y2 - y1) / height
        rows.append((cls_id, xc, yc, bw, bh))
    return rows


def save_inference_preview(model_path: Path, dataset_dir: Path, run_root: Path, conf: float = CONF, sample_size: int = SAMPLE_SIZE, seed: int = SEED) -> Optional[Path]:
    try:
        import cv2
        import matplotlib.pyplot as plt
    except ImportError as exc:
        print(f'[WARN] Skipping inference preview because optional package is missing: {exc}')
        return None

    _, test_split = split_names(dataset_dir)
    image_dir = dataset_dir / test_split / 'images'
    label_dir = dataset_dir / test_split / 'labels'
    images = [p for p in iter_images(image_dir) if (label_dir / f'{p.stem}.txt').exists()]
    if not images:
        print('[WARN] No test images with labels available for inference preview.')
        return None

    rng = random.Random(seed)
    samples = rng.sample(images, min(sample_size, len(images)))
    model = YOLO(str(model_path))

    cols = 2
    rows_count = len(samples)
    fig, axes = plt.subplots(rows_count, cols, figsize=(14, max(4, rows_count * 3.2)))
    if rows_count == 1:
        axes = [axes]

    detections = {}
    for row_axes, image_path in zip(axes, samples):
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        gt_rows = read_label_file(label_dir / f'{image_path.stem}.txt')
        result = model.predict(str(image_path), conf=conf, verbose=False)[0]
        pred_rows = predictions_as_yolo_rows(result)

        detections[image_path.name] = [
            {'class_id': cls_id, 'class_name': CLASS_NAMES.get(cls_id, f'cls_{cls_id}'), 'xywh': [xc, yc, bw, bh]}
            for cls_id, xc, yc, bw, bh in pred_rows
        ]

        row_axes[0].imshow(draw_yolo_boxes(image_rgb, gt_rows, CLASS_NAMES, 'GROUND TRUTH'))
        row_axes[0].axis('off')
        row_axes[0].set_title(image_path.name, fontsize=8)
        row_axes[1].imshow(draw_yolo_boxes(image_rgb, pred_rows, CLASS_NAMES, 'PREDICTION'))
        row_axes[1].axis('off')
        pred_names = [CLASS_NAMES.get(cls_id, f'cls_{cls_id}') for cls_id, *_ in pred_rows]
        row_axes[1].set_title(', '.join(pred_names) or 'no detection', fontsize=8)

    out_png = run_root / 'inference_gt_vs_pred.png'
    out_json = run_root / 'inference_predictions.json'
    plt.tight_layout()
    fig.savefig(out_png, dpi=120, bbox_inches='tight')
    plt.show()
    out_json.write_text(json.dumps(detections, indent=2))
    print(f'[OK] Inference preview written: {out_png}')
    print(f'[OK] Inference json written: {out_json}')
    return out_png

## 7. Audit, Clean, dan Tulis YAML

In [ ]:
source_report = audit_dataset(SOURCE_DATASET_DIR)
print_audit(source_report)
save_audit_report(source_report, RUN_ROOT / 'source')

source_has_invalid = report_has_invalid_labels(source_report)
prepare_dataset = bool(REMAP) or (source_has_invalid and CLEAN_INVALID_LABELS)

DATASET_DIR = create_prepared_dataset(
    SOURCE_DATASET_DIR,
    RUN_ROOT,
    REMAP,
    clean_invalid=(source_has_invalid and CLEAN_INVALID_LABELS),
).resolve() if prepare_dataset else SOURCE_DATASET_DIR

report = audit_dataset(DATASET_DIR)
print_audit(report)
save_audit_report(report, RUN_ROOT)

bad_splits = [
    split for split, info in report.items()
    if not split.startswith('_') and (info['invalid_ids'] or info['invalid_rows'])
]
if bad_splits:
    raise RuntimeError(f'Dataset masih memiliki label invalid di split: {bad_splits}')

DATA_YAML = write_data_yaml(DATASET_DIR, RUN_ROOT)
VAL_SPLIT, TEST_SPLIT = split_names(DATASET_DIR)

## 8. Visual Audit Label per Class

In [ ]:
if not SKIP_PREVIEW:
    label_preview_path = save_label_preview(DATASET_DIR, RUN_ROOT, SAMPLE_SIZE, SEED)
else:
    label_preview_path = None
    print('Preview label dilewati karena SKIP_PREVIEW=True')

## 9. Training

In [ ]:
def choose_weights() -> str:
    candidates = [
        ROOT_DIR / 'yolo26n.pt',
        ROOT_DIR / 'backend' / 'yolo26n.pt',
        ROOT_DIR / 'inference-service' / 'yolo26n.pt',
        ROOT_DIR / 'yolov8n.pt',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return 'yolov8n.pt'

WEIGHTS = choose_weights()
print(f'weights={WEIGHTS}')
print(f'epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, patience={PATIENCE}, freeze={FREEZE}, device={DEVICE}')

TRAIN_OK = False
TRAIN_RESULTS = None
try:
    model = YOLO(WEIGHTS)
    TRAIN_RESULTS = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        batch=BATCH,
        imgsz=IMGSZ,
        device=DEVICE,
        patience=PATIENCE,
        freeze=FREEZE,
        project=str(RUN_ROOT / 'runs'),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        augment=True,
        mosaic=1.0,
        mixup=0.05,
        copy_paste=0.05,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
        degrees=5.0,
        scale=0.5,
        erasing=0.3,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        save_period=10,
        plots=True,
        seed=SEED,
        verbose=True,
    )
    TRAIN_OK = True
    print('Training selesai.')
except KeyboardInterrupt:
    print('Training dihentikan manual. Checkpoint terakhir:', RUN_DIR / 'weights' / 'last.pt')
except Exception as exc:
    print(f'Training error: {exc}')
    traceback.print_exc()

## 10. Evaluasi

In [ ]:
best_pt = RUN_DIR / 'weights' / 'best.pt'
last_pt = RUN_DIR / 'weights' / 'last.pt'
eval_pt = best_pt if best_pt.exists() else last_pt

if not eval_pt.exists():
    print('Tidak ada checkpoint untuk evaluasi.')
else:
    try:
        if TRAIN_OK and TRAIN_RESULTS is not None:
            metrics = getattr(TRAIN_RESULTS, 'results_dict', {}) or {}
            map50 = metrics.get('metrics/mAP50(B)', 0)
            map5095 = metrics.get('metrics/mAP50-95(B)', 0)
            prec = metrics.get('metrics/precision(B)', 0)
            rec = metrics.get('metrics/recall(B)', 0)
        else:
            eval_model = YOLO(str(eval_pt))
            val_results = eval_model.val(data=str(DATA_YAML), device=DEVICE, verbose=False)
            map50 = val_results.box.map50
            map5095 = val_results.box.map
            prec = val_results.box.mp
            rec = val_results.box.mr

        print(f'Checkpoint : {eval_pt}')
        print(f'mAP50      : {map50:.4f}')
        print(f'mAP50-95   : {map5095:.4f}')
        print(f'Precision  : {prec:.4f}')
        print(f'Recall     : {rec:.4f}')
    except Exception as exc:
        print(f'Eval error: {exc}')
        traceback.print_exc()

try:
    from IPython.display import Image as IPImg, display
    for name in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
        path = RUN_DIR / name
        if path.exists():
            print(f'--- {name} ---')
            display(IPImg(str(path), width=780))
except Exception as exc:
    print(f'Preview plot dilewati: {exc}')

## 11. Inference Preview Ground Truth vs Prediction

In [ ]:
if eval_pt.exists() and not SKIP_PREVIEW:
    save_inference_preview(eval_pt, DATASET_DIR, RUN_ROOT, CONF, SAMPLE_SIZE, SEED)
elif not eval_pt.exists():
    print('Inference preview dilewati karena checkpoint belum ada.')
else:
    print('Inference preview dilewati karena SKIP_PREVIEW=True')

## 12. Export Model

In [ ]:
def export_checkpoint(checkpoint: Path, export_name: str = EXPORT_NAME) -> List[Path]:
    if IS_KAGGLE:
        export_dirs = [Path('/kaggle/working')]
    else:
        export_dirs = [ROOT_DIR / 'backend' / 'models', ROOT_DIR / 'model' / 'New']

    exported = []
    for export_dir in export_dirs:
        export_dir.mkdir(parents=True, exist_ok=True)
        dst = export_dir / export_name
        shutil.copy2(checkpoint, dst)
        exported.append(dst)
        print(f'[OK] Exported checkpoint: {dst} ({dst.stat().st_size / 1e6:.1f} MB)')
    return exported

if eval_pt.exists():
    exported_paths = export_checkpoint(eval_pt, EXPORT_NAME)
else:
    exported_paths = []
    print('Export dilewati karena checkpoint belum ada.')

if IS_KAGGLE:
    print(f'Download file dari Kaggle Output: {EXPORT_NAME}')

## Troubleshooting

| Masalah | Solusi |
|---|---|
| Dataset tidak ditemukan | Lokal: pastikan `dataset/Rust Corrosion Detection.v16i.yolo26/` atau zip ada. Kaggle: Add Data dataset rust/corrosion. |
| Class terlihat tertukar | Isi `REMAP` di Cell 2, contoh `{0: 3, 3: 0}`, lalu run ulang Cell 7 ke bawah. |
| Label invalid masih ada | Pastikan `CLEAN_INVALID_LABELS=True`, lalu run ulang Cell 7. |
| OOM GPU | Turunkan `BATCH=8` atau `IMGSZ=512`. |
| Hasil rendah | Cek `label_audit_by_class.png`, tambah epoch, atau gunakan weights yang lebih besar jika VRAM cukup. |